In [1]:
from IPython.display import display, HTML, Math, Markdown
display(HTML("<style>.container { width:95% !important; }</style>"))

%load_ext autoreload
%autoreload 2
from knockouts import *
import pandas
from build import parse_diet,constrain_diet
import cobra
diet = -pandas.read_csv("./diets/AGORADiets.txt",index_col=1,sep=' ')[["WesternDiet"]]
diet.columns = ["lb"]
diet_dct = parse_diet(diet)

In [2]:
MeanTCounts = pandas.read_csv("./ibdmdb/metaT_per_diagnosis.csv",index_col=0)
NormalizedTCounts = MeanTCounts.div(MeanTCounts.sum())
NormalizedTCounts.head()

,nonIBD,UC,CD,IBD
#FeatureID,,,,
HMPREF9436_RS14285,0.078333,0.039315,0.037492,0.037492
HMPREF9436_RS14685,0.023592,0.008788,0.013451,0.013451
BILO145876EF_RS04385,0.022452,0.003468,0.005717,0.005717
FAEPRAM212_RS14865,0.017201,0.023327,0.030562,0.030562
BSFG_RS23450,0.010633,0.011724,0.010224,0.010224


In [11]:
# condition = "nonIBD"
condition = "IBD"

In [12]:
log_format = '%(asctime)s %(message)s' #%(clientip)-15s %(user)-8s
import logging
from coralme.builder.main import ListHandler
log = logging.getLogger()
logging.basicConfig(filename = "./ibdmdb/m_integration/integration_{}.log".format(condition), filemode = 'w', level = logging.WARNING, format = log_format)
logging.captureWarnings(True)

In [13]:
survivors = set(pandas.read_csv("survivors.txt",index_col=0,header=None).index.to_list())
done = set(i.split("_killable")[0] for i in os.listdir("./ibdmdb/m_integration/{}".format(condition)) if "txt" in i)
run_for = survivors - done
len(run_for)

0

In [6]:
def run(org):
    ListHandler.print_and_log("Loading {}".format(org))
    model = cobra.io.json.load_json_model('./agora-models/AGORA_2_01_json_renamed/{}.json'.format(org))
    constrain_diet(model,diet_dct)
    kill_genes = get_m_targets(model,condition,NormalizedTCounts)
    pandas.DataFrame(columns=kill_genes).T.to_csv("./ibdmdb/m_integration_targets/{}/{}_targets.txt".format(condition,org),header=None)
    
    killable = get_m_killable(model,kill_genes,ListHandler=ListHandler,org=org)
    
    pandas.DataFrame(columns=killable).T.to_csv("./ibdmdb/m_integration/{}/{}_killable.txt".format(condition,org),header=None)

In [ ]:
# run(list(run_for)[0])

In [ ]:
NP = min([8,len(run_for)])
pool = mp.Pool(NP,maxtasksperchild=1)
pbar = tqdm(total=len(run_for),position=0,leave=True)
pbar.set_description('Building ({} threads)'.format(NP))
def collect_result(result):
    pbar.update(1)

for org in run_for:
    args = ([org])
    pool.apply_async(run,args, callback=collect_result)
pool.close()
pool.join()

In [ ]:
# l1 = set("RNA_"+i for i in pandas.read_csv("./ibdmdb/integration/nonIBD/Abiotrophia_defectiva_ATCC_49176_killable.txt",index_col=0).index)
# l2 = set(pandas.read_csv("./ibdmdb/integration/nonIBD/_Abiotrophia_defectiva_ATCC_49176_killable.txt",index_col=0).index)
# l3 = set("RNA_"+i for i in pandas.read_csv("./ibdmdb/reduction/nonIBD/Abiotrophia_defectiva_ATCC_49176_killable.txt",index_col=0).index)